In [1]:
! pip install ISLP
from ISLP import load_data

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.3/349.3 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 522.0/522.0 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.2/815.2 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 926.4/926.4 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.5/94.5 kB 5.7 MB/s eta 0:00:00
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4031 sha256=ac7f537e69fcf0f8191283f18805ff6992f9de29d96450c0b32f53c9bcc80553
  Stored in directory: /root/.cache/pip/wheels/25/cc/e0/ef2969164144c899fedb22b338f6703e2b9cf46eeebf254991
Successfully built autograd-gamma
  Attempting un

Solution 7. In this problem, you will use support vector approaches in order to
predict whether a given car gets high or low gas mileage based on the
Auto data set.
(a) Create a binary variable that takes on a 1 for cars with gas
mileage above the median, and a 0 for cars with gas mileage
below the median.
Scale the features to mean 0 and standard deviation 1 before training

In [5]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Load the dataset
url = 'https://www.statlearning.com/s/Auto.csv'
data = pd.read_csv(url)

# Treat "?" as NaN, drop rows with missing values, and convert 'horsepower' to integer
data['horsepower'] = data['horsepower'].replace('?', np.nan)
data.dropna(inplace=True)
data['horsepower'] = data['horsepower'].astype(int)  # Convert 'horsepower' to int after handling NaNs

### Part (A): Create a binary variable for high mileage
# Calculate the median of 'mpg'
median_mpg = data['mpg'].median()

# Create a new column 'high_mpg' where 1 indicates mpg > median, otherwise 0
data['high_mpg'] = np.where(data['mpg'] > median_mpg, 1, 0)

# Separate features (X) and target (y)
# Exclude 'mpg' and 'high_mpg' from features as they are the target-related columns
X = data.drop(columns=['mpg', 'high_mpg'])
y = data['high_mpg']

# Convert categorical variables, if any, to numerical using one-hot encoding
X = pd.get_dummies(X, drop_first=True)

# Scale the features to mean 0 and standard deviation 1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

(b) Fit a support vector classifer to the data with various values of
C, in order to predict whether a car gets high or low gas mileage.
Report the cross-validation errors associated with diferent values of this parameter. Comment on your results. Note you will
need to ft the classifer without the gas mileage variable to produce sensible results.                           

                                          
For part b, use a linear kernel SVC
For both b and c, use the F1-score to evaluate the SVC’s performance
You may treat origin and year as continuous variables for this problem

In [7]:
# Import necessary libraries for model training and evaluation
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import make_scorer, f1_score
import pandas as pd
from sklearn.preprocessing import StandardScaler

### Part (B): Train a Support Vector Classifier (SVC) with a Linear Kernel and Various Values of C
#Set up the Support Vector Classifier with a linear kernel
svc_linear = SVC(kernel='linear')

#Define the parameter grid for the regularization parameter 'C'
param_grid = {'C': [0.001, 0.01, 0.1, 1, 5, 10, 100]}

#Define F1-score as the evaluation metric
f1_scorer = make_scorer(f1_score)

#Set up KFold cross-validation with 5 folds and shuffling
kfold = KFold(n_splits=5, random_state=42, shuffle=True)  # Use random_state only if shuffle=True

#Perform grid search with cross-validation to find the best value for 'C'
grid_search = GridSearchCV(estimator=svc_linear, param_grid=param_grid, scoring=f1_scorer, cv=kfold)
grid_search.fit(X_scaled, y)

#Report cross-validation errors (1 - mean cross-validated F1 score) for each value of C
print("Cross-validation errors for different values of C (1 - F1-score):")
for mean_score, params in zip(grid_search.cv_results_['mean_test_score'], grid_search.cv_results_['params']):
    print(f"C={params['C']}: Cross-validation error = {(1 - mean_score):.4f} | F1-score = {mean_score:.4f}")

#Output the best parameter for 'C' and best cross-validated F1-score
print("\nBest parameter for linear kernel:", grid_search.best_params_)
print("Best cross-validated F1-score:", grid_search.best_score_)

Cross-validation errors for different values of C (1 - F1-score):
C=0.001: Cross-validation error = 0.1152 | F1-score = 0.8848
C=0.01: Cross-validation error = 0.1009 | F1-score = 0.8991
C=0.1: Cross-validation error = 0.0973 | F1-score = 0.9027
C=1: Cross-validation error = 0.1113 | F1-score = 0.8887
C=5: Cross-validation error = 0.1218 | F1-score = 0.8782
C=10: Cross-validation error = 0.1217 | F1-score = 0.8783
C=100: Cross-validation error = 0.1308 | F1-score = 0.8692

Best parameter for linear kernel: {'C': 0.1}
Best cross-validated F1-score: 0.9027292370139033


### **Comment on the Results for Part B (Linear Kernel SVC)**

#### **1. Cross-Validation Errors and F1-Scores for Different Values of `C`**

The cross-validation errors and F1-scores for different values of the regularization parameter `C` are as follows:

| C     | Cross-Validation Error | F1-Score |
|-------|------------------------|----------|
| 0.001 | 0.1152                 | 0.8848   |
| 0.01  | 0.1009                 | 0.8991   |
| 0.1   | 0.0973                 | 0.9027   |
| 1     | 0.1113                 | 0.8887   |
| 5     | 0.1218                 | 0.8782   |
| 10    | 0.1217                 | 0.8783   |
| 100   | 0.1308                 | 0.8692   |

#### **2. Interpretation of Results**

- **Best Parameter (`C = 0.1`)**:
    - The best value for the regularization parameter `C` is **`0.1`**, which yields the highest **F1-score** of **`0.9027`**.
    - This means that, for this dataset and linear kernel, a moderate value of `C` (neither too small nor too large) provides the best trade-off between maximizing the margin and minimizing classification errors.

- **Impact of `C` on Model Performance**:
    - **Small `C` values (e.g., `C = 0.001, C = 0.01`)**:
        - With smaller values of `C`, the model allows more misclassifications but maintains a wider margin between classes.
        - As seen from the results, the F1-scores for smaller values of `C` are relatively high (e.g., **F1-score = 0.8848** for `C = 0.001`), but they are slightly lower than the best result.
    
    - **Optimal `C = 0.1`**:
        - The F1-score peaks at **`C = 0.1`**, meaning this value strikes the best balance between margin width and classification accuracy.
        - At this point, the model is neither too lenient (small `C`) nor too strict (large `C`) in penalizing misclassifications.

    - **Larger `C` values (e.g., `C = 5, C = 10, C = 100`)**:
        - As `C` increases, the model becomes more strict about minimizing classification errors, leading to a narrower margin.
        - However, this can cause overfitting to the training data, resulting in lower generalization performance on unseen data.
        - For example, with **`C = 100`**, the F1-score drops to **`0.8692`**, indicating that overfitting may be occurring.

#### **3. Trade-Off Between Margin Width and Misclassification Penalty**

- The parameter `C` controls the trade-off between maximizing the margin and minimizing classification errors:
    - A smaller value of `C` allows a wider margin but tolerates more misclassifications.
    - A larger value of `C` forces the model to classify more points correctly but may lead to overfitting by creating a narrower margin.

#### **4. Generalization Performance**

- The best cross-validated F1-score is **`0.9027`**, which indicates that the model performs well on this dataset with a linear kernel SVC.
- The relatively small difference in F1-scores across different values of `C` suggests that the model is robust and not overly sensitive to changes in regularization strength within this range.

#### **5. Conclusion**

- The optimal value of `C = 0.1` gives the best balance between bias and variance for this linear kernel SVC.
- The model achieves a high F1-score (**90%**) with this setting, indicating good performance in distinguishing between cars with high and low mileage based on the selected features.
  
- For future improvements, you could explore non-linear kernels (e.g., RBF or polynomial) to see if they provide better performance by capturing more complex relationships in the data.


Part (C): Train Support Vector Classifier (SVC) with RBF and Polynomial Kernels

In [8]:
# Step 1: Define the parameter grids for both RBF and Polynomial kernels
param_grid_rbf = {
    'C': [0.001, 0.01, 0.1, 1, 5, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1]
}
param_grid_poly = {
    'C': [0.001, 0.01, 0.1, 1, 5, 10, 100],
    'degree': [2, 3, 4],
    'gamma': ['scale', 'auto', 0.001, 0.01]
}

# Step 2: Define F1-score as the evaluation metric
f1_scorer = make_scorer(f1_score)

# Step 3: Set up KFold cross-validation with 5 folds and shuffling
kfold = KFold(n_splits=5, random_state=42, shuffle=True)

# Step 4: Perform grid search with cross-validation to find the best parameters for RBF kernel
print("Training with RBF kernel...")
grid_search_rbf = GridSearchCV(estimator=SVC(kernel='rbf'), param_grid=param_grid_rbf, scoring=f1_scorer, cv=kfold)
grid_search_rbf.fit(X_scaled, y)

# Report the results for the RBF kernel
print("\nRBF Kernel - Cross-validation errors for different values of C and gamma:")
for mean_score, params in zip(grid_search_rbf.cv_results_['mean_test_score'], grid_search_rbf.cv_results_['params']):
    print(f"C={params['C']}, gamma={params['gamma']}: Cross-validation error = {(1 - mean_score):.4f} | F1-score = {mean_score:.4f}")

# Step 5: Perform grid search with cross-validation to find the best parameters for Polynomial kernel
print("\nTraining with Polynomial kernel...")
grid_search_poly = GridSearchCV(estimator=SVC(kernel='poly'), param_grid=param_grid_poly, scoring=f1_scorer, cv=kfold)
grid_search_poly.fit(X_scaled, y)

# Report the results for the Polynomial kernel
print("\nPolynomial Kernel - Cross-validation errors for different values of C, degree, and gamma:")
for mean_score, params in zip(grid_search_poly.cv_results_['mean_test_score'], grid_search_poly.cv_results_['params']):
    print(f"C={params['C']}, degree={params['degree']}, gamma={params['gamma']}: Cross-validation error = {(1 - mean_score):.4f} | F1-score = {mean_score:.4f}")

# Step 6: Output the best parameters and best cross-validated F1-scores for both kernels
print("\nBest parameters for RBF kernel:", grid_search_rbf.best_params_)
print("Best cross-validated F1-score for RBF kernel:", grid_search_rbf.best_score_)

print("\nBest parameters for Polynomial kernel:", grid_search_poly.best_params_)
print("Best cross-validated F1-score for Polynomial kernel:", grid_search_poly.best_score_)

Training with RBF kernel...

RBF Kernel - Cross-validation errors for different values of C and gamma:
C=0.001, gamma=scale: Cross-validation error = 0.4793 | F1-score = 0.5207
C=0.001, gamma=auto: Cross-validation error = 0.4793 | F1-score = 0.5207
C=0.001, gamma=0.001: Cross-validation error = 0.4533 | F1-score = 0.5467
C=0.001, gamma=0.01: Cross-validation error = 0.4818 | F1-score = 0.5182
C=0.001, gamma=0.1: Cross-validation error = 0.4777 | F1-score = 0.5223
C=0.01, gamma=scale: Cross-validation error = 0.4793 | F1-score = 0.5207
C=0.01, gamma=auto: Cross-validation error = 0.4793 | F1-score = 0.5207
C=0.01, gamma=0.001: Cross-validation error = 0.4533 | F1-score = 0.5467
C=0.01, gamma=0.01: Cross-validation error = 0.4818 | F1-score = 0.5182
C=0.01, gamma=0.1: Cross-validation error = 0.4777 | F1-score = 0.5223
C=0.1, gamma=scale: Cross-validation error = 0.4793 | F1-score = 0.5207
C=0.1, gamma=auto: Cross-validation error = 0.4793 | F1-score = 0.5207
C=0.1, gamma=0.001: Cross-v

### **Comment on the Results for Part C (RBF and Polynomial Kernels)**

#### **1. RBF Kernel Results**

- **Best Parameters**: The best parameters for the RBF kernel were found to be **`C = 100`** and **`gamma = 0.001`**.
- **Best Cross-Validated F1-Score**: The highest F1-score achieved with these parameters was **0.9042**, which is a strong performance.

##### **Analysis of RBF Kernel Results:**
- **Small `C` Values**: For small values of `C` (e.g., `C = 0.001`, `C = 0.01`), the cross-validation errors are high, and the F1-scores are low (around **0.52 - 0.55**). This indicates that the model is underfitting, as it allows a wide margin at the cost of more misclassifications.
  
- **Optimal `C = 100` with `gamma = 0.001`**: The best performance is achieved with a relatively large value of `C = 100`, which penalizes misclassifications more heavily, and a small value of `gamma = 0.001`. This combination allows the model to capture more complex relationships in the data while maintaining a good generalization capability.
  
- **Effect of Gamma (`gamma`)**:
    - When `gamma` is set to very small values like `0.001`, the model performs well because it considers points farther from the decision boundary, leading to smoother decision boundaries.
    - Larger values of `gamma`, such as `0.1`, tend to overfit the data, resulting in lower F1-scores (e.g., **F1-score = 0.5270** for `C = 100`, `gamma = 0.1`).

##### **Conclusion for RBF Kernel**:
- The RBF kernel performs very well with an F1-score of **0.9042**, which is comparable to or better than the linear kernel results from Part B.
- The best combination of hyperparameters (`C = 100`, `gamma = 0.001`) suggests that a strong regularization (high `C`) and a smooth decision boundary (small `gamma`) provide the best balance between bias and variance.

#### **2. Polynomial Kernel Results**

- **Best Parameters**: The best parameters for the polynomial kernel were found to be **`C = 1`**, **`degree = 2`**, and **`gamma = scale`**.
- **Best Cross-Validated F1-Score**: The highest F1-score achieved with these parameters was only **0.5659**, which is significantly lower than the performance of both the linear and RBF kernels.

##### **Analysis of Polynomial Kernel Results:**
- **Low Performance Across Parameter Values**:
    - Most combinations of parameters for the polynomial kernel yield relatively low F1-scores, ranging from around **0.43 to 0.56**.
    - Even with different degrees (`degree = 2, 3, or 4`) and various values for gamma (`scale`, `auto`, or small values like `0.001`), the polynomial kernel struggles to achieve high performance.

- **Effect of Degree (`degree`)**:
    - Increasing the degree of the polynomial kernel does not seem to improve performance significantly.
    - For example, with `degree = 3`, the F1-score remains around **0.5612**, and with higher degrees like `degree = 4`, performance tends to degrade further.

##### **Conclusion for Polynomial Kernel**:
- The polynomial kernel does not perform as well as either the linear or RBF kernels on this dataset.
- Even at its best (with `C = 1`, `degree = 2`, and `gamma = scale`), it achieves an F1-score of only **0.5659**, indicating that this kernel may not be suitable for this particular problem.

#### **3. Overall Comparison**

| Kernel       | Best Parameters                                          | Best F1-Score |
|--------------|----------------------------------------------------------|---------------|
| Linear       | C = 0.1                                                  | 0.9027        |
| RBF          | C = 100, gamma = 0.001                                   | 0.9042        |
| Polynomial   | C = 1, degree = 2, gamma = scale                         | 0.5659        |

##### **Key Takeaways**:
- The **RBF kernel** slightly outperforms both the linear and polynomial kernels with an F1-score of **0.9042**, suggesting that it captures non-linear relationships in the data better than a simple linear decision boundary.
  
- The **polynomial kernel** performs poorly compared to both linear and RBF kernels, indicating that it may not be well-suited for this dataset or task.

#### **4. Conclusion**
The results show that while both linear and RBF kernels perform well on this dataset, with F1-scores around or above **90%**, the RBF kernel has a slight edge in terms of performance due to its ability to model non-linear relationships effectively. On the other hand, the polynomial kernel struggles to achieve competitive performance, making it less suitable for this problem.
`